In [1]:
import os
import sys
import socket
import xarray as xr
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import re
from datetime import datetime, timedelta, date
from pathlib import Path
import xesmf as xe
import time


In [2]:
def get_python_path():
    hostname = socket.gethostname()                                 # 1. Identify the computer by hostname
    code_locations = {                                              # 2. Set default Python code location based on hostname
        "NECMAC04363461.local": "/Users/kimberly.hyde/Documents/",  # Mac laptop
        "nefscsatdata": "/mnt/EDAB_Archive/",                       # Satdata
        "guihyde": "/mnt/EDAB_Archive/"                             # Kim's Satdata container
    }

    base_path = code_locations.get(hostname)
    if not base_path:
        print(f"Unknown hostname: {hostname}")
        return None

    default_utility_path = Path(base_path) / "nadata/python"
    if not default_utility_path.is_dir():
        print(f"Directory not found: {default_utility_path}")
        return None

    print(f"Default utilities path: {default_utility_path}")
    return default_utility_path

python_path = get_python_path()
if str(python_path) not in sys.path:
    sys.path.insert(0, str(python_path))

from utilities import date_utilities, gridding_utilities, file_utilities, import_utilities, calc_daylength


Default utilities path: /Users/kimberly.hyde/Documents/nadata/python


In [4]:
def daily_vgpm(date, chl, sst, par, lat, lon, output_dir,input_dirs):
    #d8 = str(date).split('T')[0]
    
    # Define the output file and check if exists and is up to date
    output_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
    should_process = True
    
    
    if os.path.exists(output_file):
        if not file_make(input_files, output_file):
    #    if not any_input_newer(input_dirs, date, output_file):
    #        expected = chl.sel(time=str(date)).time.dt.floor("D").values
    #    if not missing_dates(output_file, expected):
            print(f"✓ Skipping {date} — output file exists.")
            return
    #        should_process = False
    #if not should_process:

    print(f"The output file is: {output_file}")
    
    # Subset dataset to the specified year
    chl_day = chl.sel(time=str(date))
    sst_day = sst.sel(time=str(date))
    par_day = par.sel(time=str(date))

    # Broadcast DOY and LAT for day length calculation
    doy = chl_day['time'].dt.dayofyear
    doy_3d, lat_3d = xr.broadcast(doy, lat)

    # Calculate the day length
    daylength = day_length_calculation(lat_3d.values, doy_3d.values)
    
    # Convert day_length to 2D array
    daylength = np.tile(daylength.reshape(-1, 1), (1, len(lon)))
    
    # Convert day_length to dataset
    day_length = xr.DataArray(
        data=daylength,
        dims=chl_day.dims,
        coords=chl_day.coords,
        name="day_length"
    )
    
    # Change variables to float32
    chl_day = chl_day.astype("float32")
    sst_day = sst_day.astype("float32")
    par_day = par_day.astype("float32")
    day_length = day_length.astype("float32")
   
    print("Running vgpm_models...")
    pp_eppley, pp_vgpm, kdpar, chl_eu, zeu =vgpm_models(chl_day,sst_day,par_day,day_length)
    
    
    #Create data arrays for each variable
    xa_pp_eppley = xr.DataArray(pp_eppley,dims=chl_day.dims,coords=chl_day.coords,name="pp_eppley")
    xa_pp_vgpm = xr.DataArray(pp_vgpm,dims=chl_day.dims,coords=chl_day.coords,name="pp_vgpm")
    xa_kdpar = xr.DataArray(kdpar,dims=chl_day.dims,coords=chl_day.coords,name="pp_kdpar")
    xa_chl_euphotic = xr.DataArray(chl_eu,dims=chl_day.dims,coords=chl_day.coords,name="chl_euphotic")
    xa_zeu = xr.DataArray(zeu,dims=chl_day.dims,coords=chl_day.coords,name="zeu")
    
    
    # Create a dataset
    pp_dataset = xr.Dataset({
        "pp_eppley": xa_pp_eppley,
        "pp_vgpm": xa_pp_vgpm,
        "kdpar": xa_kdpar,
        "chl_euphotic": xa_chl_euphotic,
        "zeu": xa_zeu
    })
    

    pp_dataset = pp_dataset.expand_dims({"time":[date]})
    
    print(f"✓ Writing {output_file}")
    pp_dataset.to_netcdf(output_file)

In [5]:
#1 Find CHL, SST and PAR file paths

from utilities import get_prod_files
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)
input_dirs = [chl_path,sst_path,par_path]

chl_files = get_prod_files('chl',dataset='OCCCI')
sst_files = get_prod_files('sst',dataset='CORALSST')
par_files = get_prod_files('par',dataset='GLOBCOLOUR')

#2 For each CHL file find the SST and PAR files with the matching date
from utilities import get_source_file_dates
chl_dates = get_source_file_dates(chl_files)
par_dates = get_source_file_dates(par_files)
sst_dates = get_source_file_dates(sst_files)

#3 Check if the output file exists or if the input data are newer - skip if don't need to process

#4 Regrid the SST and PAR files to match the CHL grid

#5 Find the "valid" data for all input data

#6 Get the day length for all "valid" data points

#7 Run the VGPM model on the valid data

#8 Save the output model

📦 Found 30 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL
📦 Found 32 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST
📦 Found 31 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR


In [ ]:
from utilities import parse_dataset_info
from utilities import get_prod_files

chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
fp = parse_dataset_info(chl_path)
ppd_path = chl_path
ppd_path = ppd_path.replace(fp.dataset_type,'OUTPUT_DATA')
ppd_path = ppd_path.replace(fp.product,'PPD')
ppd_path

{'dataset': 'OCCCI', 'version': 'V6.0', 'dataset_type': 'SOURCE_DATA', 'dataset_map': 'MAPPED_4KM_DAILY', 'product': 'CHL'}


In [ ]:
from utilities import make_product_output_dir

def load_daily_inputs(date, chl_path, sst_path, par_path):
    """
    Load CHL, SST, and PAR datasets for a given date.

    Parameters:
        date (str): Date string (e.g. '2025-03-10') for logging/debugging.
        chl_path, sst_path, par_path (str): File paths to input NetCDFs.
        chunks (dict, optional): Chunking strategy for xarray (e.g. {"lat": 100, "lon": 100})

    Returns:
        tuple: 
    """
    
    chl = xr.open_dataset(chl_path)#, chunks={"lat": 100, "lon": 100})
    sst = xr.open_dataset(sst_path)#, chunks={"lat": 100, "lon": 100})
    par = xr.open_dataset(par_path)#, chunks={"lat": 100, "lon": 100})
    return chl, sst, par


#def process_one_date(date, paths, lat, lon, output_dir, input_dirs):
#    chl_path, sst_path, par_path = paths
#    chl, sst, par = load_daily_inputs(date, chl_path, sst_path, par_path)
#    daily_vgpm(date, chl, sst, par, lat, lon, output_dir, input_dirs)


def process_one_date(date, paths, output_dir, input_dirs):
    """
    Processes a single day's CHL, SST, PAR maps to compute VGPM globally.

    Parameters:
        date (str): Date in "YYYY-MM-DD" format.
        paths (tuple): (chl_path, sst_path, par_path)
        output_dir (str): Directory to save results.
        input_dirs (dict): Optional extra data sources, e.g. ancillary files.

    Returns:
        None
    """

    chl_path, sst_path, par_path = paths

    
    

    out_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
    if os.path.exists(out_file):
        if not file_make(paths, out_file):
            print(f"✓ Skipping {date} — output file exists.")
            return
        

    # Check if out_file exists and if so, if the input data are newer than the out_file

    try:
        chl_ds = xr.open_dataset(chl_path)
        sst_ds = xr.open_dataset(sst_path)
        par_ds = xr.open_dataset(par_path)

        chl = chl_ds["chlor_a"]
        sst = sst_ds["analysed_sst"]
        par = par_ds["PAR_mean"]

        # Regrid using xarray-regrid
        if verbose:
            print("Regridding...")
        sst = sst.regrid.linear(chl_ds)
        par = par.regrid.linear(chl_ds)

        # Assuming lat/lon grids are identical and present in CHL file
        lat = chl_ds["lat"].values
        lon = chl_ds["lon"].values

        # Mask valid pixels
        valid_mask = ~np.isnan(chl) & ~np.isnan(sst) & ~np.isnan(par)

        result_array = np.full(chl.shape, np.nan)

        # Loop through valid pixels
        for i, j in zip(*np.where(valid_mask)):
            chl_val = chl[i, j]
            sst_val = sst[i, j]
            par_val = par[i, j]
            lat_val = lat[i]
            lon_val = lon[j]

            result_array[i, j] = vgpm_models(chl_val, sst_val, par_val, lat_val, lon_val, date)

        # Save result
        out_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
        result_ds = xr.Dataset(
            {
                "VGPM": (["lat", "lon"], result_array)
            },
            coords={
                "lat": lat,
                "lon": lon
            },
            attrs={"generated_by": "Copilot process_one_date"}
        )
        result_ds.to_netcdf(out_file)

        print(f"✅ {date}: processed {np.sum(valid_mask)} pixels → {out_file}")

    except Exception as e:
        print(f"❌ {date}: failed with error → {e}")



In [ ]:
from utilities import file_make
from utilities import regrid_wrapper


def process_daily_pp(date, chl_path, sst_path, par_path, output_dir, regridder, daylength_cache_dir):
    """
    Process daily CHL, SST, and PAR inputs:
    - Checks if output exists or needs update
    - Regrids SST and PAR
    - Computes or loads daylength
    - Saves processed output
    """
    # 1️⃣ Construct output path and check if processing is needed
    
    
    
    output_path = file_make(date=date, source="CHL_SST_PAR", output_dir=output_dir)
    if not output_path:
        print(f"✅ Skipping {date} — output already up-to-date.")
        return

    # 2️⃣  Load CHL directly (assumed to be on target grid)
    chl = xr.mfopen_dataset(chl_path)

    # 3️⃣ Regrid SST and PAR
    daterange = (date, date)
    sst_ds = regrid_wrapper(chl_path,sst_path,source_vars=['analysed_sst'],daterange=daterange)
    par_ds = regrid_wrapper(chl_path,par_path,source_vars=['PAR_mean'],daterange=daterange)
    

    # 4️⃣ Compute or load daylength
    doy = pd.Timestamp(date).dayofyear
    latitudes = chl["lat"].values  # Assuming CHL grid is target
    daylength_ds = get_or_compute_daylength(doy, latitudes, daylength_cache_dir)

    # 5️⃣ Combine and save
    processed = xr.Dataset({
        "chl": chl["chlor_a"],
        "sst": sst_rg["sst"],
        "par": par_rg["par"],
        "daylength": daylength_ds["daylength"]
    })
    processed.to_netcdf(output_path)
    print(f"💾 Saved: {output_path}")


In [11]:

def build_pp_date_map(dates=None, get_date_prod="CHL", chl_dataset=None, sst_dataset=None, par_dataset=None):
    """
    Constructs a date→(CHL, SST, PAR, PPD) file path mapping using get_prod_files.

    Parameters:
        dates (list of str): List of dates (YYYYMMDD) to process.
        get_date_prod (str): Which product to use for date filtering.
        chl_dataset, sst_dataset, par_dataset (str): Dataset identifiers.   
    Returns:
        dict: Mapping of date → (chl_path, sst_path, par_path,ppd_output_path)
    """
    
    import os, re
    from utilities import get_source_file_dates, get_prod_files, get_dates, make_product_output_dir,parse_dataset_info
    
    #def extract_date_from_path(path):
    #    match = re.search(r"(\d{8})", os.path.basename(path))
    #    return f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}" if match else None
    
    # Get files
    chl_files = get_prod_files("CHL", dataset=chl_dataset)
    sst_files = get_prod_files("SST", dataset=sst_dataset)
    par_files = get_prod_files("PAR", dataset=par_dataset)
    
    # Extract metadata from first CHL file
    if not chl_files:
        raise ValueError("No CHL files found to derive metadata.")
    info = parse_dataset_info(chl_files[0])

    # Extract valid date strings
    chl_dates = get_source_file_dates(chl_files, format="yyyymmdd", placeholder=None)
    sst_dates = get_source_file_dates(sst_files, format="yyyymmdd", placeholder=None)
    par_dates = get_source_file_dates(par_files, format="yyyymmdd", placeholder=None)
    
    # Construct the ppd file names
    #ppd_dates = chl_dates
    #ppd_files = f"D_{ppd_dates}-{info['dataset']}-{info['version']}-{info['dataset_map']}-PPD.nc"

    # Build reverse maps (date → file)
    chl_map = {d: f for d, f in zip(chl_dates, chl_files) if d}
    sst_map = {d: f for d, f in zip(sst_dates, sst_files) if d}
    par_map = {d: f for d, f in zip(par_dates, par_files) if d}
    #ppd_map = {d: f for d, f in zip(ppd_dates, ppd_files) if d}
    # Build PPD file map
    output_dir = make_product_output_dir('CHL','PPD',dataset=chl_dataset)
    ppd_map = {}
    for date in chl_dates:
        filename = f"D_{date}-{info['dataset']}-{info['version']}-{info['dataset_map']}-PPD.nc"
        ppd_map[date] = os.path.join(output_dir, filename)

    # Normalize get_date_prod
    prod_key = (get_date_prod or "CHL").upper()

    # Get date list
    target_dates = get_dates(dates)
    if target_dates is None:
        if prod_key == "CHL":
            target_dates = sorted(chl_map.keys())
        elif prod_key == "SST":
            target_dates = sorted(sst_map.keys())
        elif prod_key == "PAR":
            target_dates = sorted(par_map.keys())
        else:
            raise ValueError(f"Invalid get_date_prod value: '{get_date_prod}'")

     # Build mapping for dates present in CHL and all other inputs
    pp_data_map = {}
    for date in target_dates:
        if date in chl_map and date in sst_map and date in par_map:
            pp_data_map[date] = (chl_map[date], sst_map[date], par_map[date], ppd_map[date])
        else:
            print(f"⚠ Missing data for {date}: CHL {'✅' if date in chl_map else '❌'}  SST {'✅' if date in sst_map else '❌'}  PAR {'✅' if date in par_map else '❌'}")

    print(f"📅 Mapped {len(pp_data_map)} dates with complete product files.")
    return pp_data_map


In [13]:
pmap = build_pp_date_map(get_date_prod="CHL", sst_dataset='CORALSST')
pmap


📦 Found 30 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL
📦 Found 32 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST
📦 Found 31 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR
📅 Mapped 30 dates with complete product files.


{'19980101': ('/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980101-fv6.0.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST/coraltemp_v3.1_19980101.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980101__GLOB_4_AV-SWF_PAR_DAY_00.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/OUTPUT_DATA/MAPPED_4KM_DAILY/PPD/D_19980101-OCCCI-V6.0-MAPPED_4KM_DAILY-PPD.nc'),
 '19980102': ('/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980102-fv6.0.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST/coraltemp_v3.1_19980102.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/G

In [6]:

def run_vgpm_batch(date_paths_map, output_dir, input_dirs):
    with ProcessPoolExecutor() as executor:
        futures = []
        for date, paths in date_paths_map.items():
            futures.append(executor.submit(
                process_one_date, date, paths, output_dir, input_dirs
            ))
        for f in futures:
            f.result()


In [ ]:
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)
input_dirs = [chl_path,sst_path,par_path]
input_dirs

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def process_one_date(date, paths, lat, lon, output_dir, input_dirs):
    chl_path, sst_path, par_path = paths
    chl, sst, par = load_daily_inputs(date, chl_path, sst_path, par_path)
    daily_vgpm(date, chl, sst, par, lat, lon, output_dir, input_dirs)


In [ ]:
date_path_map = build_pp_date_map(sst_dataset='CORALSST')

chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)
input_dirs = [chl_path,sst_path,par_path]

def run_vgpm_batch(date_paths_map, lat, lon, output_dir, input_dirs):
    with ProcessPoolExecutor() as executor:
        futures = []
        for date, paths in date_paths_map.items():
            futures.append(executor.submit(
                process_one_date, date, paths, lat, lon, output_dir, input_dirs
            ))
        for f in futures:
            f.result()  # Optional: catch exceptions


💡 Optional Enhancements
If you know all grids are regular, you can precompute and cache regrid weights using xESMF for even faster execution. This will precompute the interpolation mapping once, so it doesn’t get recalculated every time you regrid. This can save a ton of time if you’re regridding many variables or looping over many files.
Validate units and fill values before regridding to avoid cross-variable inconsistencies (e.g. 0.0 vs NaN vs masked arrays).
Add logging with simple markers like ✅ SST regridded to keep processing transparent.

🚀 How It Works
xESMF separates two steps:
Build the regridder: This creates a weight matrix based on source and target grids.
Apply the regridder: You reuse that weight matrix to quickly regrid each variable.

🧪 Example
import xesmf as xe

# Create regridder from SST grid → CHL grid
regridder = xe.Regridder(sst_ds, chl_ds, method="bilinear", reuse_weights=True)

# Apply to SST and PAR variables
sst = regridder(sst_ds["analysed_sst"])
par = regridder(par_ds["PAR_mean"])

reuse_weights=True tells xESMF to save/load the weights file in the background (in a .nc file).
You can control where it saves with filename="my_weights.nc" if you need portability.

🧠 Why It Matters for You
Batch Efficiency: If chl_ds is fixed across time, just create one regridder and reuse across all dates.
Parallel-Friendly: Each process/thread avoids recomputing weights independently.
Debuggability: You can inspect the generated weights file to understand interpolation behavior (e.g., land/sea masking).

Feature	xesmf.Regridder(method="bilinear")	xarray-regrid.linear()
Backend	ESMF (Earth System Modeling Framework)	Pure Python (NumPy + xarray)
Interpolation Method	Bilinear via ESMF weight matrix	Bilinear via coordinate-based interpolation
Grid Support	Rectilinear & curvilinear	Mostly rectilinear (lat/lon)
Weight Caching	✅ Supports .nc weight reuse	❌ No weight caching
Performance	Fast with cached weights; slower to build	Lightweight; fast for small grids
Missing Value Handling	Explicit masking via ESMF	NaNs propagate unless manually handled
Parallelization	Dask-compatible	Dask-compatible

🧠 When to Use Which?
Use xesmf if:
You need high-precision regridding across complex or curvilinear grids
You want to reuse weights across time steps or variables
You care about masking, extrapolation, or conservative methods
Use xarray-regrid.linear() if:
You want a lightweight, dependency-free solution
Your grids are simple and small
You’re prototyping or debugging quickly

🧪 Accuracy & Behavior
In benchmark comparisons:
xesmf bilinear tends to produce smoother and more numerically stable results, especially when upscaling or working near masked regions.
xarray-regrid is great for quick interpolation, but may diverge slightly in edge cases or when extrapolation is needed.

## Compare the xarray_regrid to the xesmf bileaner regrid

In [ ]:
import xarray as xr
import xesmf as xe
import xarray_regrid as xrgrid
import matplotlib.pyplot as plt

# Get file paths
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)

# Load sample source data
chl_ds = xr.open_mfdataset(os.path.join(chl_path,'*.nc'))
chl = chl_ds['chlor_a']
chl = chl.isel(time=0)

# Build target grid from CHL
chl_grid = xr.Dataset({
    "lat": chl_ds["lat"],
    "lon": chl_ds["lon"]
})

# Get SST data that needs to be regridded
sst_ds = xr.open_mfdataset(os.path.join(sst_path,'*.nc'))
sst = sst_ds["analysed_sst"]
sst = sst.isel(time=0)

# Load sample source data
par_ds = xr.open_mfdataset(os.path.join(par_path,'*.nc'))
par = par_ds['PAR_mean']
par = par.isel(time=0)

In [ ]:

def suggest_chunks(target_grid, max_chunk=1000):
    lat_size = len(target_grid["lat"])
    lon_size = len(target_grid["lon"])
    lat_chunk = min(lat_size, max_chunk)
    lon_chunk = min(lon_size, max_chunk)
    return {"lat": lat_chunk, "lon": lon_chunk}


# xesmf bilinear
regridder_xesmf = xe.Regridder(sst, chl_grid, "bilinear", reuse_weights=False)
data_xesmf = regridder_xesmf(sst)

output_chunks = suggest_chunks(chl_grid)
data_xesmf = data_xesmf.chunk(output_chunks)


In [ ]:
print("Chunk structure:", data_xesmf.chunks)
print("Shape:", data_xesmf.shape)


In [ ]:
data_xesmf.isel(time=0).plot()


In [ ]:

# xarray-regrid linear
data_regrid = sst.regrid.linear(chl_ds)


In [ ]:
data_regrid.isel(time=0).plot()

In [ ]:
import time

t0 = time.time()
data_xesmf = data_xesmf.compute()
print("Computed xesmf:", time.time() - t0, "s")

t1 = time.time()
data_regrid = data_regrid.compute()
print("Computed regrid:", time.time() - t1, "s")


In [ ]:
t2 = time.time()
diff = data_xesmf - data_regrid
print("Computed diff:", time.time() - t2, "s")


print("Mean difference:", float(diff.mean()))
print("Max absolute difference:", float(abs(diff).max()))


In [ ]:
# Plot first
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
data_xesmf.plot(ax=axs[0], cmap="viridis", robust=True)
axs[0].set_title("xesmf bilinear")
data_regrid.plot(ax=axs[1], cmap="viridis", robust=True)
axs[1].set_title("xarray-regrid linear")
plt.tight_layout()
plt.show()

# Then compute diff
diff = data_xesmf - data_regrid
print("Mean difference:", float(diff.mean()))


In [ ]:

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
data_xesmf.plot(ax=axs[0], cmap="viridis", robust=True)
axs[0].set_title("xesmf bilinear")

data_regrid.plot(ax=axs[1], cmap="viridis", robust=True)
axs[1].set_title("xarray-regrid linear")

plt.tight_layout()
plt.show()

diff = data_xesmf - data_regrid
print("Mean difference:", float(diff.mean()))
print("Max absolute difference:", float(abs(diff).max()))


In [ ]:
xr.open_dataset("/Users/kimberly.hyde/Documents/nadata/python/resources/regrid_weights/regrid_weights_OCCCI_MAPPED_4KM_DAILY_CHL_76cb42dc6fbc29e57a135f07f802533e_to_CORALSST_MAPPED_5KM_DAILY_SST_ae4a3dc043f63d2dd80dd0d80f20f4c8_withgrids.nc").data_vars

In [ ]:
import xesmf as xe

# Create regridder from SST grid → CHL grid
regridder = xe.Regridder(sst_ds, chl_ds, method="bilinear", reuse_weights=True)

# ??? Does xe.Regridder "bilenear" return the same result as regrid.linear???

# Apply to SST and PAR variables
sst = regridder(sst_ds["analysed_sst"])
par = regridder(par_ds["PAR_mean"])


In [ ]:
# Load your dataset (must have lat/lon as 2D arrays if irregular)
dirdata = r'/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/'
dirchl = os.path.join(dirdata,'OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/')
ds = xr.open_mfdataset(os.path.join(dirchl,'*.nc'), combine='by_coords')

area_da = calc_mapped_pixel_area(ds['lat'].values, ds['lon'].values)

In [ ]:
def has_grids(weights_file):
    """Checks if the weights file has the lon lat grids"""
    ds = xr.open_dataset(weights_file)
    return all(k in ds for k in ['src_lon', 'src_lat', 'dst_lon', 'dst_lat'])

def ensure_regrid_grids(weights_file, src_ds, tgt_ds):
    """
    Patch grid coordinates into a weights file if missing.
    Uses src_ds and tgt_ds directly for safety and cross-version compatibility.
    """
    
    # Create a temp path and copy of the file
    tmp_weights_file = weights_file.replace(".nc", "_tmp.nc")
    shutil.copy(weights_file, tmp_weights_file)
    
    # Check the temporary file copy
    patched = xr.open_dataset(tmp_weights_file)

    expected = {'src_lat', 'src_lon', 'dst_lat', 'dst_lon'}
    missing = expected - set(patched.data_vars)

    if not missing:
        print("✅ Grid variables already present:", expected)
        os.remove(tmp_weights_file)
        return

    print(f"🔧 Missing grid variables: {missing}. Patching into {weights_file}...")

    # Add the source and target coordinates to the temp file
    patched["src_lat"] = src_ds["lat"]
    patched["src_lon"] = src_ds["lon"]
    patched["dst_lat"] = tgt_ds["lat"]
    patched["dst_lon"] = tgt_ds["lon"]

    patched.to_netcdf(weights_file)
    print("✅ Grids patched successfully.")
    if os.path.exists(tmp_weights_file):
        os.remove(tmp_weights_file)